# Inspect room alignment

Source and GT use **mesh vertex colors** when present (reconstructed GLB). CAD without vertex colors uses USD `displayColor` or gray. Clicked pairs stay highlighted in orange.

Overlay defaults to `TELEOP_FLAGS` (`--xyz-deg` / `--xyz` / `--scale`) applied to the **original** native cloud. Paste a full teleop line or just those flags. Clicking source-to-GT pairs is optional: a 7-DoF similarity (Umeyama, same as scaled ICP) is fit to those pairs. `SOURCE_FLIP_X/Y/Z` and `GT_FLIP_X/Y/Z` are **picker display only** (they are not baked into `--xyz-deg`). A picker-derived teleop command is that fit plus the GLB Y-up→Z-up Rx(+90°) so Isaac applies it to the native cloud. Scale is always ≥ 0. Picking uses subsampled points. Source and GT picker clouds use `SOURCE_CROP_*` / `GT_CROP_*` in the import cell.

Needs Open3D, plotly, anywidget, ipywidgets, and pxr (Isaac Sim `python.sh` or `usd-core`) in this kernel. Plotly 6 `FigureWidget` (the picker) requires `anywidget` (`pip install anywidget`). The overlay is a plain Plotly figure so it works in VS Code / Cursor Jupyter. Re-run the import cell after helper edits; if plotly was imported before installing anywidget, restart the kernel first.


In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd().resolve()
if (HERE / "inspect_alignment.py").is_file():
    SCRIPTS = HERE
    REPO = HERE.parents[1]
else:
    REPO = Path("/home/ubuntu/workspace/ebim-benchmark")
    SCRIPTS = REPO / "task2_isaacsim" / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

import importlib
import inspect_alignment as viz

importlib.reload(viz)

TELEOP_FLAGS = (
    "--xyz-deg 89.4646 1.3006 1.54099 "
    "--xyz 2.01734 2.003 1.70 "
    "--scale 1.18892"
)
N_POINTS = 80_000
PICKER_MAX_POINTS = 20_000
SHOW_GT = True
SHOW_PREVIOUS = False
SHOW_TRANSFORMED = True
GT_CROP_X = (0.0, 7.0)
GT_CROP_Y = (0.0, 4.0)
GT_CROP_Z = (2.0, 10.0)
GT_CROP_Z = (-10, 10.0)
# SOURCE_CROP_X = (-7.0, -0.0)
SOURCE_CROP_X = (-3.0, 2.0)
SOURCE_CROP_Y = (-2.0, 2.0)
# SOURCE_CROP_Y = (-20, 20.0)
# SOURCE_CROP_Z = (0.5, 10.0)
SOURCE_CROP_Z = (-10, 10.0)
GT_FLIP_X, GT_FLIP_Y, GT_FLIP_Z = False, False, False
SOURCE_FLIP_X, SOURCE_FLIP_Y, SOURCE_FLIP_Z = False, True, True
# Picker display-only (X/Y/Z). Not baked into --xyz-deg.
GT_FLIP_AXIS = [GT_FLIP_X, GT_FLIP_Y, GT_FLIP_Z]
SOURCE_FLIP_AXIS = [SOURCE_FLIP_X, SOURCE_FLIP_Y, SOURCE_FLIP_Z]

## Echo modern office

Collider GLB **mesh** vs CAD GT **mesh**. The overlay cell applies `TELEOP_FLAGS` to the original native cloud. Click corresponding points (source then GT) only if you want to refit; a near-identity picker fit prints `--xyz-deg` near **+90**. The `.usdz` is only the NuRec volume for the teleop command.


In [ ]:
echo_mesh = (
    REPO / "Echo_Modern_office_space_with_cubicles.reconstructed_mesh.glb"
)
echo_usd = REPO / "Echo_Modern_office_space_with_cubicles.usdz"
echo = viz.load_meshes(echo_mesh)
print(
    f"meshes  gt={len(echo['gt_mesh'].triangles)} tris  "
    f"src={len(echo['source_mesh'].triangles)} tris"
)

In [ ]:
echo = viz.apply_command_alignment(echo, TELEOP_FLAGS)
print(f"scale={echo['scale']:.4f}  t={echo['translation']}")
print(viz.teleop_command(echo, echo_usd))
viz.overlay(
    echo["gt_mesh"],
    echo["source_mesh_native"],
    echo["transformed_mesh"],
    show_gt=SHOW_GT,
    show_previous=False,
    show_transformed=SHOW_TRANSFORMED,
    crop_gt=False,
    max_triangles=None,
)

## Optional: click correspondences

Skip unless you want to refit. Click source then GT; the next cell replaces `TELEOP_FLAGS` with the Umeyama fit.

In [ ]:
src, gt = viz.picker_points(
    echo,
    max_points=PICKER_MAX_POINTS,
    crop_x=GT_CROP_X,
    crop_y=GT_CROP_Y,
    crop_z=GT_CROP_Z,
    source_crop_x=SOURCE_CROP_X,
    source_crop_y=SOURCE_CROP_Y,
    source_crop_z=SOURCE_CROP_Z,
    gt_flip_axis=GT_FLIP_AXIS,
    source_flip_axis=SOURCE_FLIP_AXIS,
)
print(
    f"picker points  src={len(src.xyz)}  gt={len(gt.xyz)}  "
    f"src_rgb={src.rgb is not None}  gt_rgb={gt.rgb is not None}"
)
picker = viz.CorrespondencePicker(
    src.xyz, gt.xyz, source_rgb=src.rgb, gt_rgb=gt.rgb
)
picker.widget()

In [ ]:
fit = picker.solve()
# Optional refit: replaces TELEOP_FLAGS. Flips are picker-only;
# the command is T_fit @ GLB Rx(+90°) on the native cloud.
echo = viz.apply_manual_alignment(
    echo,
    fit["transformation"],
    source_flip_axis=SOURCE_FLIP_AXIS,
    gt_flip_axis=GT_FLIP_AXIS,
)
print(f"scale={echo['scale']:.4f}  t={echo['translation']}")
print(viz.teleop_command(echo, echo_usd))
# Validate the teleop command on the original native cloud (no picker flips).
viz.overlay(
    echo["gt_mesh"],
    echo["source_mesh_native"],
    echo["transformed_mesh"],
    show_gt=SHOW_GT,
    show_previous=False,
    show_transformed=SHOW_TRANSFORMED,
    crop_gt=False,
    max_triangles=None,
)

## Optional: ICP instead of clicks

`run_alignment` still runs scaled ICP on sampled clouds. Skip unless you want a comparison.

In [ ]:
echo_icp = viz.run_alignment(echo_mesh, n_points=N_POINTS)
print(f"scale={echo_icp['scale']:.4f}  t={echo_icp['translation']}")
print(viz.teleop_command(echo_icp, echo_usd))
viz.overlay(
    echo_icp["gt_mesh"],
    echo_icp["source_mesh_native"],
    echo_icp["transformed_mesh"],
    show_gt=SHOW_GT,
    show_previous=SHOW_PREVIOUS,
    show_transformed=SHOW_TRANSFORMED,
    crop_gt=False,
    max_triangles=None,
)

In [ ]:
import numpy as np

Z_MIN = 0.5

src = echo["source_mesh"]
n_verts_before = len(np.asarray(src.vertices))
n_tris_before = len(np.asarray(src.triangles))

cropped = viz.crop_mesh_xy(
    src,
    x_range=None,
    y_range=None,
    z_range=(Z_MIN, 1e9),
    require_all=True,
)
native = viz._as_open3d_mesh(cropped)
native.transform(np.linalg.inv(viz.align.Y_UP_TO_Z_UP_COLUMN))

out_path = echo_mesh.with_name(
    f"{echo_mesh.stem}_groundplane_removed{echo_mesh.suffix}"
)
o3d = viz.align._require_open3d()
if not o3d.io.write_triangle_mesh(
    str(out_path), native, write_vertex_colors=True
):
    raise RuntimeError(f"failed to write {out_path}")

print(
    f"ground plane z<{Z_MIN}:  "
    f"verts {n_verts_before} -> {len(native.vertices)}  "
    f"tris {n_tris_before} -> {len(native.triangles)}"
)
print(out_path)
